In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
 
from scipy.stats import norm
from scipy.optimize import minimize

In [2]:
# ----------------------------- config ---------------------------------------
RNG_SEED = 42

vars_keep_input = [
    'Fairbanks_T2_1d_max_ensmean', 'Fairbanks_T2_1h_max_ensmean', 'Fairbanks_T2_30d_max_ensmean', 'Fairbanks_T2_7d_max_ensmean', 
    'Fort_Bragg_PRECT_1d_max_ensmean', 'Fort_Bragg_PRECT_3d_max_ensmean', 'Fort_Bragg_PRECT_5d_max_ensmean', 
    'Fort_Bragg_SPEI_09_ensmean_mean', 'Fort_Bragg_SPEI_09_ensmean_min', 'Fort_Bragg_SPEI_48_ensmean_mean', 'Fort_Bragg_SPEI_48_ensmean_min', 
    'Guam_SPEI_03_ensmean_mean', 'Guam_SPEI_03_ensmean_min', 'Guam_SPEI_48_ensmean_mean', 'Guam_SPEI_48_ensmean_min',
    'Pituffik_FT_days_ensmean', 'Pituffik_MDD_ensmean', 
    'Yuma_PG_PRECT_1d_max_ensmean', 'Yuma_PG_PRECT_7d_max_ensmean', 
    'Yuma_PG_SPEI_24_ensmean_mean', 'Yuma_PG_SPEI_24_ensmean_min', 'Yuma_PG_SPEI_48_ensmean_mean', 'Yuma_PG_SPEI_48_ensmean_min', 'Yuma_PG_days_above_999p_ensmean'
]

vars_keep_target = [
    'Fairbanks_T2_1d_max', 'Fairbanks_T2_1h_max', 'Fairbanks_T2_30d_max', 'Fairbanks_T2_7d_max', 
    'Fort_Bragg_PRECT_1d_max', 'Fort_Bragg_PRECT_3d_max', 'Fort_Bragg_PRECT_5d_max', 'Fort_Bragg_SPEI_09_mean', 'Fort_Bragg_SPEI_09_min', 'Fort_Bragg_SPEI_48_mean', 'Fort_Bragg_SPEI_48_min', 
    'Guam_SPEI_03_mean', 'Guam_SPEI_03_min', 'Guam_SPEI_48_mean', 'Guam_SPEI_48_min', 
    'Pituffik_FT_days', 'Pituffik_MDD', 
    'Yuma_PG_PRECT_1d_max', 'Yuma_PG_PRECT_7d_max', 'Yuma_PG_SPEI_24_mean', 'Yuma_PG_SPEI_24_min', 'Yuma_PG_SPEI_48_mean', 'Yuma_PG_SPEI_48_min', 'Yuma_PG_days_above_999p'
]

years_train = np.arange(1959, 2000)
years_verif = np.arange(2000, 2021)

data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
in_input  = os.path.join(data_dir, 'STN_CESM_ALL.zarr')
in_target = os.path.join(data_dir, 'STN_ERA5_ALL.zarr')

# ----------------------------- load -----------------------------------------
ds_input = xr.open_zarr(in_input)[vars_keep_input].drop_vars('valid_year', errors='ignore')
ds_input_train = ds_input.sel(init_year=years_train)
ds_input_verif = ds_input.sel(init_year=years_verif)
 
ds_target = xr.open_zarr(in_target)[vars_keep_target]

n_lead   = ds_input_train.sizes['lead_year']
n_train  = ds_input_train.sizes['init_year']
n_verif  = ds_input_verif.sizes['init_year']
 
lead_years        = ds_input_train['lead_year'].values
init_years_train  = ds_input_train['init_year'].values
init_years_verif  = ds_input_verif['init_year'].values


# ================================== #
# AnEn
# ================================== #
# ds_input has int64 dims of (init_year, lead_year, member), computed from ensemble mean; valid_year is defined as init_year+lead_year
# ds_target has int64 dims of (valid_year)
# Complete AnEn search based on ds_input_train and ds_target, and each variable and lead_year separately
# Identify the optimal number of AnEn members, save them as ds_dress

In [3]:
K_candidates = np.array([5, 10, 20], dtype=np.int64)
K_out        = 20  # divisible by every K_candidate
# K_candidates = np.array([5, 10, 15, 20, 25, 30], dtype=np.int64)
# K_out        = int(np.lcm.reduce(K_candidates))   # 300

# ------------------------ helpers --------------------------------------------
def _align_xy(da_in, da_tgt, init_years, lead):
    """Aligned (x, y) for one (var, lead), dropping pairs missing either side.
    x : SMYLE ensemble-mean predictor               — shape (n,)
    y : ERA5 verification at valid = init + lead    — shape (n,)
    """
    x = da_in.sel(init_year=init_years, lead_year=lead).values
    valid_years = init_years + int(lead)
    avail = da_tgt['valid_year'].values
    y = np.full(len(init_years), np.nan, dtype=np.float64)
    in_mask = np.isin(valid_years, avail)
    if in_mask.any():
        y[in_mask] = da_tgt.sel(valid_year=valid_years[in_mask]).values
    keep = np.isfinite(x) & np.isfinite(y)
    return x[keep], y[keep]

def _crps_ens(ens, y):
    """Fair CRPS, ens shape (n, K), y shape (n,). Returns shape (n,)."""
    t1   = np.mean(np.abs(ens - y[:, None]), axis=1)
    diff = np.abs(ens[:, :, None] - ens[:, None, :])
    t2   = 0.5 * diff.mean(axis=(1, 2))
    return t1 - t2

# ------------------------ main loop ------------------------------------------
dress_arr = {v: np.full((n_verif, n_lead, K_out), np.nan, dtype=np.float64)
             for v in vars_keep_input}
K_opt     = {v: np.full(n_lead, -1, dtype=np.int64)
             for v in vars_keep_input}

K_grid_max = int(K_candidates.max())

for var_in, var_tgt in tqdm(list(zip(vars_keep_input, vars_keep_target)),
                            desc='AnEn fit/apply  '):
    da_in_tr = ds_input_train[var_in]
    da_in_vf = ds_input_verif[var_in]
    da_tgt   = ds_target[var_tgt]

    for il, lead in enumerate(lead_years):
        # --- training pool aligned to ERA5 ---------------------------------
        x_tr, y_tr = _align_xy(da_in_tr, da_tgt, init_years_train, lead)
        if len(x_tr) <= K_grid_max:
            continue
        sigma = float(np.std(x_tr, ddof=1))
        if not np.isfinite(sigma) or sigma == 0.0:
            continue

        # --- K selection via leave-one-out CV on training pool -------------
        d_tr = np.abs(x_tr[:, None] - x_tr[None, :]) / sigma
        np.fill_diagonal(d_tr, np.inf)                  # exclude self
        neigh_tr = y_tr[np.argsort(d_tr, axis=1)[:, :K_grid_max]]  # (n_tr, K_max)

        best_K, best_score = -1, np.inf
        for K in K_candidates:
            score = float(np.mean(_crps_ens(neigh_tr[:, :K], y_tr)))
            if score < best_score:
                best_score, best_K = score, int(K)
        K_opt[var_in][il] = best_K

        # --- apply to verification period ----------------------------------
        x_vf      = da_in_vf.sel(lead_year=lead).values   # (n_verif,)
        vf_finite = np.isfinite(x_vf)
        if not vf_finite.any():
            continue

        d_vf = np.abs(x_vf[vf_finite, None] - x_tr[None, :]) / sigma
        top  = np.argsort(d_vf, axis=1)[:, :best_K]
        ens  = y_tr[top]                                            # (n_vf_ok, K_best)

        # tile K_best analogs to K_out (each repeated K_out/K_best times)
        reps = K_out // best_K
        dress_arr[var_in][vf_finite, il, :] = np.tile(ens, (1, reps))

AnEn fit/apply  : 100%|██████████| 24/24 [00:03<00:00,  7.54it/s]


In [4]:
# ------------------------ wrap into xarray -----------------------------------
ds_dress_p = xr.Dataset(
    {v: (('init_year', 'lead_year', 'member'), dress_arr[v])
     for v in vars_keep_input},
    coords={
        'init_year': init_years_verif,
        'lead_year': lead_years,
        'member':    np.arange(K_out),
    },
    attrs={'description': 'AnEn-dressed ERA5-valued ensemble (verification period).',
           'K_candidates': K_candidates.tolist(),
           'K_out': K_out},
)

ds_K_opt = xr.Dataset(
    {v: (('lead_year',), K_opt[v]) for v in vars_keep_input},
    coords={'lead_year': lead_years},
    attrs={'description': 'K_best per (variable, lead_year) chosen by LOOCV CRPS.'},
)

In [5]:
# ============================================================================
# Verification
# ============================================================================
def lookup_target(y_full, valid_years_needed, valid_years_avail):
    out = np.full(len(valid_years_needed), np.nan, dtype=np.float64)
    in_mask = np.isin(valid_years_needed, valid_years_avail)
    if in_mask.any():
        out[in_mask] = y_full.sel(valid_year=valid_years_needed[in_mask]).values
    return out

def crps_ensemble(ens, y):
    out = np.full(ens.shape[0], np.nan)
    finite_ens = np.all(np.isfinite(ens), axis=1)
    mask = finite_ens & np.isfinite(y)
    if not mask.any():
        return out
    e = ens[mask]; yv = y[mask]
    term1 = np.mean(np.abs(e - yv[:, None]), axis=1)
    diff  = np.abs(e[:, :, None] - e[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    out[mask] = term1 - term2
    return out
 
 
def anomaly_correlation(fc, obs):
    m = np.isfinite(fc) & np.isfinite(obs)
    if m.sum() < 2:
        return np.nan
    fa = fc[m]  - np.mean(fc[m])
    oa = obs[m] - np.mean(obs[m])
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan
 
 
verif_rows = []
for i_var, var in enumerate(tqdm(vars_keep_target, desc='Verification    ')):
    var_in = vars_keep_input[i_var]
    
    y_full = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    x_raw   = ds_input_verif[var_in].values
    x_dress = ds_dress_p[var_in].values
 
    for il, lead in enumerate(lead_years):
        valid_yrs = init_years_verif + int(lead)
        y_arr = lookup_target(y_full, valid_yrs, valid_years_avail)
 
        crps_dress = crps_ensemble(x_dress[:, il, :], y_arr)
        dress_mean = np.nanmean(x_dress[:, il, :], axis=1)
        acc_dress = anomaly_correlation(dress_mean, y_arr)
        mean_dress = float(np.nanmean(crps_dress))
        
        verif_rows.append({
            'variable':       var,
            'lead_year':      int(lead),
            'n_eff':          int(np.isfinite(y_arr).sum()),
            'CRPS_dress':     mean_dress,
            'ACC_dress':      acc_dress,
        })
 
df_verif = pd.DataFrame(verif_rows)

Verification    : 100%|██████████| 24/24 [00:01<00:00, 22.70it/s]


In [7]:
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/AnEn/'
# ds_dress_p.to_zarr(save_dir+'Dress.zarr', mode='w')

In [32]:
print("\n=== Verification summary (mean across lead_year) ===")
print(df_verif.groupby('variable')[['CRPS_dress', 'ACC_dress']].mean().round(3))


=== Verification summary (mean across lead_year) ===
                         CRPS_dress  ACC_dress
variable                                      
Fairbanks_T2_1d_max           0.867     -0.054
Fairbanks_T2_1h_max           1.118     -0.058
Fairbanks_T2_30d_max          0.832      0.059
Fairbanks_T2_7d_max           0.804     -0.005
Fort_Bragg_PRECT_1d_max      21.106      0.013
Fort_Bragg_PRECT_3d_max      10.751     -0.017
Fort_Bragg_PRECT_5d_max       6.938     -0.020
Fort_Bragg_SPEI_09_mean       0.521      0.055
Fort_Bragg_SPEI_09_min        0.579     -0.041
Fort_Bragg_SPEI_48_mean       1.088      0.026
Fort_Bragg_SPEI_48_min        1.234     -0.012
Guam_SPEI_03_mean             0.390      0.014
Guam_SPEI_03_min              0.476     -0.126
Guam_SPEI_48_mean             1.197     -0.025
Guam_SPEI_48_min              1.340     -0.008
Pituffik_FT_days              3.314      0.084
Pituffik_MDD                 39.976     -0.045
Yuma_PG_PRECT_1d_max          6.107     -0.082
Yuma_P

```
=== Verification summary (mean across lead_year) ===
                         CRPS_raw  CRPS_closed  CRPSS  ACC_raw  ACC_calib
variable                                                                 
Fairbanks_T2_1d_max         4.477        0.879  0.803    0.228      0.339
Fairbanks_T2_1h_max         4.211        1.067  0.745    0.088      0.162
Fairbanks_T2_30d_max        4.934        0.813  0.835    0.139      0.088
Fairbanks_T2_7d_max         4.532        0.809  0.821    0.125      0.180
Fort_Bragg_SPEI_09_mean     0.789        0.515  0.337    0.154      0.128
Fort_Bragg_SPEI_09_min      0.903        0.563  0.364    0.236      0.038
Fort_Bragg_SPEI_48_mean     1.365        1.046  0.203    0.227      0.145
Fort_Bragg_SPEI_48_min      1.217        1.168  0.011    0.260      0.117
Pituffik_FT_days            3.731        3.402  0.085    0.016      0.055
Guam_SPEI_03_mean        0.618        0.365  0.401    0.013      0.120
Guam_SPEI_03_min         1.987        0.493  0.748   -0.066      0.031
Guam_SPEI_48_mean        1.106        0.579  0.439    0.193      0.347
Guam_SPEI_48_min         1.611        0.621  0.598    0.188      0.337
Pituffik_FT_days         3.731        3.506  0.054    0.016     -0.032
Pituffik_MDD             64.450      25.696  0.600   -0.121     -0.086
Yuma_PG_SPEI_24_mean     0.830        0.602  0.262    0.080      0.141
Yuma_PG_SPEI_24_min      0.971        0.670  0.282    0.002      0.192
Yuma_PG_SPEI_48_mean     0.821        0.705  0.128    0.262      0.338
Yuma_PG_SPEI_48_min      0.867        0.884 -0.086    0.160      0.310
Fort_Bragg_PRECT_1d_max    21.051      20.095  0.046    0.090      0.011
Fort_Bragg_PRECT_3d_max    10.634      10.464  0.017    0.080      0.033
Fort_Bragg_PRECT_5d_max     6.852       6.769  0.012    0.108     -0.009
```